# Module 9: Clothing Similarity & Style Matching
## Visual Similarity, Fashion Color Harmony & Outfit Compatibility Engine

This notebook demonstrates:
1. Loading the `StyleMatcher` engine integrated with 1,280-dimensional visual embeddings.
2. Pairwise outfit compatibility scoring combining visual similarity, color harmony, and occasion alignment.
3. Finding top visually similar garments within the same category (alternatives).
4. Cross-part outfit matching: finding matching bottoms, footwear, and accessories for a seed garment.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns

from src.embeddings import EmbeddingManager
from src.style_matcher import StyleMatcher

sns.set_theme(style="white", palette="muted")
plt.rcParams["figure.figsize"] = (14, 7)

### 1. Initialize Style Matcher Engine

In [ ]:
manager = EmbeddingManager()
matcher = StyleMatcher(embedding_manager=manager, weight_visual=0.45, weight_color=0.35, weight_usage=0.20)
print(f"Catalog Size : {len(manager):,} indexed garments")
print(f"Embedding Dim: {manager.embeddings.shape[1]}")
print(f"Scoring Weights: Visual={matcher.w_visual}, Color={matcher.w_color}, Usage={matcher.w_usage}")

### 2. Pairwise Compatibility Evaluation

We evaluate how the system scores two candidate pairings: a harmonious classic pair vs an occasion clash.

In [ ]:
# Example 1: Classic White T-shirt + Blue Jeans
sample_top = manager.index_df[manager.index_df["outfit_part"] == "top"].iloc[0]
sample_bottom = manager.index_df[manager.index_df["outfit_part"] == "bottom"].iloc[0]

breakdown = matcher.score_pairing(sample_top, sample_bottom)
print(f"Top: {sample_top.get('productDisplayName')} ({sample_top.get('baseColour')})")
print(f"Bottom: {sample_bottom.get('productDisplayName')} ({sample_bottom.get('baseColour')})")
print("\nPairing Compatibility Assessment:")
for k, v in breakdown.items():
    print(f"  {k:20s}: {v}")

### 3. Complementary Garment Recommendation

Given a seed Top garment, we query the system to find the top matching Bottoms and Shoes.

In [ ]:
seed_id = sample_top["id"]
matching_bottoms = matcher.find_compatible_garments(
    query_item_id=seed_id,
    target_part="bottom",
    top_k=4,
)

print(f"Top-4 Matching Bottoms for '{sample_top.get('productDisplayName')}':")
for r, b in enumerate(matching_bottoms, 1):
    print(f"#{r} [{b['composite_score']:.3f}] - {b['canonical_category']} | Color: {b['baseColour']} ({b['color_harmony']}) | {b['productDisplayName']}")

### 4. Visualizing Seed Item vs. Recommended Outfit Pieces

In [ ]:
fig, axes = plt.subplots(1, len(matching_bottoms) + 1, figsize=(18, 4.5))

# Display Seed
s_img = Image.open(sample_top["image_path"]).convert("RGB")
axes[0].imshow(s_img)
axes[0].set_title(f"SEED ITEM\n{sample_top['canonical_category']}\n({sample_top['baseColour']})", color="darkred", fontweight="bold", fontsize=10)
axes[0].axis("off")

# Display Matches
for idx, item in enumerate(matching_bottoms):
    m_img = Image.open(item["image_path"]).convert("RGB")
    axes[idx + 1].imshow(m_img)
    axes[idx + 1].set_title(
        f"Score: {item['composite_score']:.3f}\n{item['canonical_category']} ({item['baseColour']})\n{item['color_harmony']}",
        color="navy",
        fontsize=9,
    )
    axes[idx + 1].axis("off")

plt.suptitle(f"Style-Matched Complementary Bottoms for {sample_top['productDisplayName']}", fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

### 5. Footwear Recommendation for Complete Outfit Cohesion

In [ ]:
matching_shoes = matcher.find_compatible_garments(
    query_item_id=seed_id,
    target_part="shoes",
    top_k=4,
)

print(f"Top-4 Matching Footwear for '{sample_top.get('productDisplayName')}':")
for r, s in enumerate(matching_shoes, 1):
    print(f"#{r} [{s['composite_score']:.3f}] - {s['canonical_category']} | Color: {s['baseColour']} | {s['productDisplayName']}")